# Chapter 03B - Give words seats. Then notice who didn't get one.

In [03A](03A-training-skeleton-and-word-objectives.ipynb), we averaged the word rows and lost
their order. Let's stop doing that. Give each context position its own seat, then join the
rows in seat order before making the guess. Same lookup, different way of using the rows.

**Why switch from legal phrases to colors?** We want a question with one obvious clue:
`red` should lead to `warm`, and `blue` to `cool`. Put a few words between clue and answer.
Now we can see exactly when the model's window cuts the clue off. These same eight sentences
and the same training/test split stay with us in 03C.

First we'll try counting, which needs no learned embeddings. Then we'll train a window model,
including its word rows. Its practice loop is written out so you can see the whole experiment.
In 03C we'll use the inherited loop again. This tiny color task tests what the model can see,
not how good it is at English.

**Code-only view:** [Python companion](03B-counts-and-fixed-window-language-models.py).

# Counts and fixed windows: what can your predictor actually see?

**Guiding question:** Can a better predictor recover a clue that never reaches its input?

You know how to fit a network from [Stage 02](../../02-pytorch-foundations/README.md).
Predict the next word in `red wait quietly now warm`. A three-word window sees `wait quietly now`,
not `red`. Choose pieces by splitting at spaces. Those pieces are **tokens**, the units you count
or predict. No downloads or previous notebook state are needed.

| Attempt | Complaint | Evidence |
|---|---|---|
| Count the previous word | Histories share `now` | Next-word counts |
| Learn ordered windows | A bag forgets positions | Permutation comparison |
| Improve the same window | Color remains outside | Exactly equal predictions |

**What we'll try:** build counts and a small neural language model, meaning a next-token predictor.
Illustrate the family map; leave smoothing recipes and large-corpus quality out of scope.
[Handwritten companion](03B-counts-and-fixed-window-language-models-theory.md).

## Fix one problem, notice the next

![Counts lead to a bounded window, a running note, and a reader and writer. Looking back removes a source-access restriction.](images/03-evo-01-model-evolution-map.png)
*A route of complaints, not a ranking of newer models.*


## 1. Count what comes after `now`

**Predict:** Will seeing `now` alone tell you whether `warm` or `cool` comes next?

```text
red  wait quietly now -> warm
blue wait quietly now -> cool
                  same previous word
```

Count each neighboring pair. A **bigram** is two neighboring tokens.
A **unigram** count ignores even the previous word and counts single tokens.
These are observed counts, not a claim about all English.

In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)
started = time.perf_counter()

In [ ]:
corpus = [f"{clue} {first} {second} now {answer}".split()
          for clue, answer in [("red", "warm"), ("blue", "cool")]
          for first in ["wait", "rest"] for second in ["quietly", "calmly"]]
train_rows = [r for r in corpus if r[1:3] != ["rest", "calmly"]]
test_rows = [r for r in corpus if r[1:3] == ["rest", "calmly"]]
for row in corpus:
    print(" ".join(row), "|", "held out" if row in test_rows else "training")

In [ ]:
from collections import Counter, defaultdict
single_counts = Counter(word for row in train_rows for word in row)
next_counts = defaultdict(Counter)
for row in train_rows:
    for previous, answer in zip(row, row[1:]):
        next_counts[previous][answer] += 1
now_counts = next_counts["now"]
print("Single-token counts:", dict(single_counts))
print("After now:", dict(now_counts))
bigram_final = max(now_counts.values()) / sum(now_counts.values())
print(f"Best constant choice after now: {bigram_final:.1%}")
fig, ax = plt.subplots(figsize=(5, 2.4))
ax.bar(list(now_counts), list(now_counts.values()), color=["#2c7fb8", "#ef8a23"])
ax.set(ylabel="Observed count", title="One previous word leaves two answers")
plt.show()
print("Takeaway: now alone does not identify the earlier color clue.")

**Your turn:** Change only `previous_word`. Predict whether its table has one answer or several.
Keep the corpus unchanged: test what an input reveals before fitting another model.

In [ ]:
previous_word = "quietly"  # CHANGE THIS: try "now" or "red"
print(previous_word, "->", dict(next_counts[previous_word]))

**But there's a catch:** Each count table treats its context as a separate case.
Can a network reuse learned features while keeping the order of the words?

## 2. Give each position its own seat

**Predict:** Should `red wait quietly` and `quietly wait red` produce the same prediction?
A bag sees the same pieces. An ordered window gives each piece a seat.

```text
bag:     {red, wait, quietly}       shuffling leaves the bag unchanged
ordered: [seat 1 | seat 2 | seat 3] shuffling moves pieces between seats
```

Tensors now need integer addresses. A **token ID** is an address, not a numerical meaning.
Each address retrieves a trainable row of numbers. That row is an **embedding**.
Training adjusts the rows along with the predictor. A row is not a definition of a word.
Flattening joins rows in seat order; averaging would erase that order.
Tiny floating-point rounding gaps in the bag comparison are not recovered order.
`<start>` fills missing seats before the beginning. It is not an observed word.

In [ ]:
vocab = ["<start>"] + sorted({word for row in corpus for word in row})
to_id = {word: i for i, word in enumerate(vocab)}
def encode(rows):
    return torch.tensor([[to_id[word] for word in row] for row in rows])
train_ids, test_ids = encode(train_rows), encode(test_rows)
print("ID addresses:", to_id)

In [ ]:
WINDOW = 3
def window_pairs(rows):
    contexts, answers = [], []
    for row in rows:
        for position in range(1, len(row)):
            prefix = ["<start>"] * WINDOW + row[:position]
            contexts.append([to_id[word] for word in prefix[-WINDOW:]])
            answers.append(to_id[row[position]])
    return torch.tensor(contexts), torch.tensor(answers)
class WindowLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.rows = nn.Embedding(len(vocab), 8)
        self.predictor = nn.Sequential(nn.Linear(WINDOW * 8, 24), nn.Tanh(),
                                       nn.Linear(24, len(vocab)))
    def forward(self, context):
        return self.predictor(self.rows(context).flatten(1))

In [ ]:
model = WindowLM()
a = torch.tensor([[to_id[w] for w in ["red", "wait", "quietly"]]])
b = a.flip(1)
with torch.no_grad():
    bag_gap = (model.rows(a).mean(1) - model.rows(b).mean(1)).abs().max().item()
    ordered_gap = (model(a) - model(b)).abs().max().item()
print(f"Bag row-average gap: {bag_gap:.8f}")
print(f"Ordered score gap before training: {ordered_gap:.8f}")
print("Takeaway: seats preserve an order difference; that alone does not prove useful learning.")
x, y = window_pairs(train_rows)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
losses = []
for step in range(300):
    loss = F.cross_entropy(model(x), y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
print(f"Training loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
fig, ax = plt.subplots(figsize=(5, 2.4))
ax.plot(losses, color="#2c7fb8")
ax.set(xlabel="Update", ylabel="Next-token loss", title="Learning cannot remove missing input")
plt.show()

**Your turn:** Change one word in `probe_words`. Inspect the trained next-token choices.
A changed prediction shows input sensitivity, not broad language understanding.

In [ ]:
probe_words = ["red", "wait", "quietly"]  # CHANGE THIS: replace only "red" with "blue"
with torch.no_grad():
    probabilities = model(torch.tensor([[to_id[w] for w in probe_words]])).softmax(-1)[0]
print([(vocab[i], round(probabilities[i].item(), 3))
       for i in probabilities.topk(3).indices.tolist()])

**But there's a catch:** you fixed the lost order. You didn't move the window's left edge.

## 3. The clue is one seat too far away

**Predict:** Can more training distinguish `red rest calmly now` from `blue rest calmly now`
if the model only gets their last three words?

We kept `rest calmly` out of training. But we don't need to guess how well the model will
handle it to spot this problem: both inputs are literally `rest calmly now`.
A deterministic model must give identical inputs the same output. Good training, bad training,
no training at all: none can reveal a color that isn't there.

![Red and blue lie outside identical windows. A window-only predictor cannot distinguish their required answers.](images/03-evo-02-fixed-window-blind-spot.png)
*A predictor cannot use a clue outside its input.*


In [ ]:
contexts = test_ids[:, -4:-1]
answers = test_ids[:, -1]
with torch.no_grad():
    predictions = model(contexts).softmax(-1)
    blind_gap = (predictions[0] - predictions[1]).abs().max().item()
    window_final = (predictions.argmax(-1) == answers).float().mean().item()
print("Inputs:", [[vocab[i] for i in row] for row in contexts.tolist()])
print("Required answers:", [vocab[i] for i in answers.tolist()])
print(f"Maximum probability difference: {blind_gap:.8f}")
print(f"Held-out final-token accuracy: {window_final:.1%}")
assert torch.equal(contexts[0], contexts[1]) and blind_gap == 0
print("Takeaway: identical windows force identical predictions, not identical correct answers.")

**Your turn:** Change only `visible_words`. At four words, is the color visible?
This shows the clue has come into view. It doesn't train a four-word model: we'd need to rebuild the input layer for four seats and train it again.

In [ ]:
visible_words = 3  # CHANGE THIS: try 4
visible = test_ids[:, -(visible_words + 1):-1]
print([[vocab[i] for i in row] for row in visible.tolist()])
print("Still identical?", torch.equal(visible[0], visible[1]))

## 4. What carries over to bigger models?

| Teaching mechanism | Production counterpart | What changes | What stays |
|---|---|---|---|
| Neighbor count | Bounded n-gram predictor | More contexts, smoothing, storage | Missing context stays missing |
| Trainable rows | Token embedding table | Larger vocabulary and width | IDs retrieve learned rows |
| Ordered seats | Neural fixed-context predictor | More data and wider window | Input has a left boundary |

Use the printed results to see what happened in your run.

In [ ]:
print({"bigram_final_accuracy": bigram_final, "bag_gap": bag_gap,
       "ordered_score_gap": ordered_gap, "window_final_accuracy": window_final,
       "identical_window_probability_gap": blind_gap})
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")

### What we tried, explained, and left for later

- **What we built and checked:** whitespace tokens, unigram/bigram counts, context/answer pairs,
  IDs at first tensor use, learned rows, ordered-window training, bag permutation invariance,
  and identical-window predictions on the held-out pair.
- **What we explained:** reusable features, input boundaries, longer n-gram sparsity
  (longer exact contexts have fewer observations), and the route to recurrent readers.
- **What we're leaving for later:** smoothing (repair sparse counts), subword tokenization
  (represent new words), convolutional sequence models (another neighborhood route),
  and large-corpus benchmarks. See [tokenization](../02-tokenization-and-embeddings/README.md).

**Watch out for these traps:** do not interpret ID magnitude as meaning; do not call a bag ordered;
never blame optimization for information removed before the forward pass.

**The question we're left with:** What if you carry the color forward instead of chopping it off?
[03C](03C-fixed-windows-to-recurrent-language-models.ipynb) reuses these exact eight sentences.